<a href="https://colab.research.google.com/github/vinayaksr06/vinayaksr06/blob/main/Copy_of_Activity_1_Titanic_Dataset_Preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ACTIVITY 1: DATASET PREPARATION
**Unit I – Foundations & Data Preparation | Marks: 5**

**Dataset:** Titanic passenger dataset (Seaborn public CSV copy)  
**Domain:** Transport  
**Objective:** Inspect data quality, perform basic EDA, clean missing values, encode categorical features, scale selected numerical features, and save a prepared CSV.  
**Important:** This activity does not train a prediction model.

**Dataset source:** [Seaborn Titanic CSV on GitHub](https://github.com/mwaskom/seaborn-data/blob/master/titanic.csv)  
**Direct CSV:** `https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv`

The commonly used file contains passenger information and survival status. The dataset includes numerical and categorical columns and is suitable for demonstrating data preparation.

## 1. Import required libraries
Run this cell first. If a library is missing, install it using the command below.

In [ ]:
# If required, uncomment and run:
# !pip install pandas numpy matplotlib seaborn scikit-learn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from IPython.display import display

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

## 2. Load the dataset
The CSV is loaded directly from the public Seaborn data repository. If internet access is unavailable, download `titanic.csv` from the source link and upload it to Colab, then use `pd.read_csv('/content/titanic.csv')`.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"

try:
    df_original = pd.read_csv(DATA_URL)
except Exception as error:
    print("Automatic download failed:", error)
    from google.colab import files
    uploaded = files.upload()
    csv_name = next(iter(uploaded))
    df_original = pd.read_csv(csv_name)

print("Dataset name: Titanic passenger dataset")
print("File type: CSV")
print("Original shape (rows, columns):", df_original.shape)
display(df_original.head())

## 3. Understand the dataset and inspect its structure
Display the first and last five records, column names, data types, and dataset information.

In [ ]:
display(df_original.head())
display(df_original.tail())

print("Column names:")
print(df_original.columns.tolist())

print("\nData types and non-null counts:")
df_original.info()

print("\nDataset dimensions:", df_original.shape)

### Meaning of important columns

| Column | Meaning |
|---|---|
| `survived` | Survival status: 0 = No, 1 = Yes |
| `pclass` | Passenger ticket class: 1st, 2nd, or 3rd |
| `sex` | Passenger sex category |
| `age` | Passenger age in years |
| `sibsp` | Number of siblings/spouses aboard |
| `parch` | Number of parents/children aboard |
| `fare` | Passenger fare |
| `embarked` | Port code: C, Q, or S |
| `deck` | Deck/cabin location category, often missing |
| `embark_town` | Full name of embarkation town |

This Seaborn version also contains derived/descriptive columns such as `who`, `adult_male`, `alive`, `class`, and `alone`.

## 4. Separate numerical and categorical columns; summary statistics
`survived` is the target column for a possible future classification task. It is retained but is not scaled or used as an input feature in this activity.

In [ ]:
numeric_cols = df_original.select_dtypes(include=np.number).columns.tolist()
categorical_cols = df_original.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print("Numerical columns:", numeric_cols)
print("Categorical columns:", categorical_cols)

display(df_original[numeric_cols].describe().T)
print("\nFrequency counts for sex:")
display(df_original["sex"].value_counts(dropna=False))

## 5. Identify missing values
Calculate both the number and percentage of missing values in every column.

In [ ]:
missing_summary = pd.DataFrame({
    "Missing Count": df_original.isna().sum(),
    "Missing Percentage": (df_original.isna().mean() * 100).round(2)
}).sort_values("Missing Count", ascending=False)

display(missing_summary)
print("Columns without missing values:")
print(missing_summary[missing_summary["Missing Count"] == 0].index.tolist())

### Missing-value findings
In the standard Seaborn Titanic CSV, `age`, `deck`, `embarked`, and `embark_town` contain missing observations. The exact counts and percentages are printed above by the notebook. `deck` has a very high proportion of missing values, so it is not suitable for straightforward imputation.

## 6. Create a working copy and clean missing values
The original DataFrame remains unchanged. For this basic preparation activity:
- Fill missing `age` values with the median age within the same `pclass` and `sex` group. This uses relevant groups and is less affected by extreme values than the mean.
- Fill missing `embarked` values with its mode because it is a categorical column with very few missing entries.
- Drop `deck` because it has a large proportion of missing values.
- Drop redundant or derived columns (`alive`, `class`, `embark_town`, `who`, `adult_male`, `alone`) to avoid duplicated information. `alive` duplicates the target; `class` duplicates `pclass`; `embark_town` duplicates `embarked`.

In [ ]:
df = df_original.copy(deep=True)

# Keep track of missing values before cleaning
missing_before = df.isna().sum()

# Group-wise median imputation for age
df["age"] = df.groupby(["pclass", "sex"])["age"].transform(
    lambda series: series.fillna(series.median())
)

# Mode imputation for embarked
df["embarked"] = df["embarked"].fillna(df["embarked"].mode()[0])

# Remove a highly incomplete column and redundant/derived columns
columns_to_drop = [
    "deck", "embark_town", "alive", "class",
    "who", "adult_male", "alone"
]
df = df.drop(columns=columns_to_drop, errors="ignore")

print("Missing values before cleaning:")
display(missing_before[missing_before > 0])
print("\nMissing values after cleaning:")
display(df.isna().sum())

## 7. Check duplicates and category consistency
Only exact duplicate rows are removed. Category text is stripped of leading/trailing spaces and normalised to lowercase. The duplicate count is calculated before removal.

In [ ]:
duplicate_count = df.duplicated().sum()
print("Exact duplicate rows found:", duplicate_count)

df = df.drop_duplicates().copy()

# Standardise text categories
for col in ["sex", "embarked"]:
    df[col] = df[col].astype("string").str.strip().str.lower()

print("\nUnique values after category standardisation:")
print("sex:", sorted(df["sex"].dropna().unique().tolist()))
print("embarked:", sorted(df["embarked"].dropna().unique().tolist()))

print("\nDuplicate rows after removal:", df.duplicated().sum())

## 8. Basic exploratory data analysis (EDA)
Display updated numerical summary statistics and category frequencies. The graphs below show the age distribution and passenger counts by sex. A third graph compares survival counts by passenger class.

In [ ]:
display(df.select_dtypes(include=np.number).describe().T)

print("Passenger counts by sex:")
display(df["sex"].value_counts())

plt.figure(figsize=(8, 5))
sns.histplot(data=df, x="age", bins=25, kde=True)
plt.title("Distribution of Passenger Ages")
plt.xlabel("Age (years)")
plt.ylabel("Number of passengers")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 5))
sns.countplot(data=df, x="sex")
plt.title("Passenger Count by Sex")
plt.xlabel("Sex")
plt.ylabel("Number of passengers")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 5))
sns.countplot(data=df, x="pclass", hue="survived")
plt.title("Survival Count by Passenger Class")
plt.xlabel("Passenger class")
plt.ylabel("Number of passengers")
plt.legend(title="Survived", labels=["No (0)", "Yes (1)"])
plt.tight_layout()
plt.show()

### EDA observations (confirm after running the graphs)
1. The age histogram shows the distribution of passenger ages, with most observations concentrated among younger and middle-aged adults.
2. The sex count plot shows the number of male and female passengers; the categories are not equally represented.
3. The passenger-class plot allows comparison of survival counts across first, second, and third class. It shows that survival outcomes differ by passenger class.

Write the observations in your own words after checking the generated charts.

## 9. Encode categorical features
`sex` and `embarked` are nominal categories with no natural order, so one-hot encoding is appropriate. `pclass` is an ordinal category (1st, 2nd, 3rd) and is retained as an ordinal numeric value. The target `survived` remains 0/1.

In [ ]:
# Keep a clear list of features and target
target = "survived"
categorical_features = ["sex", "embarked"]

df_encoded = pd.get_dummies(
    df,
    columns=categorical_features,
    drop_first=False,
    dtype=int
)

print("Columns before encoding:", df.columns.tolist())
print("\nColumns after encoding:", df_encoded.columns.tolist())
display(df_encoded.head())

## 10. Scale selected numerical features
Age, family counts, and fare have different measurement scales. Standardisation transforms selected features to approximately zero mean and unit standard deviation. This can help distance-based and gradient-based algorithms. Identifier-like columns and the target are not scaled.

For a real predictive workflow, fit the scaler on the training split only. Here, the activity prepares the complete dataset for demonstration and does not train a model.

In [ ]:
scale_cols = ["age", "sibsp", "parch", "fare"]
scaler = StandardScaler()

df_prepared = df_encoded.copy()
df_prepared[scale_cols] = scaler.fit_transform(df_prepared[scale_cols])

print("Scaling method: StandardScaler (z-score standardisation)")
print("Scaled columns:", scale_cols)
display(df_prepared[scale_cols].describe().T)

## 11. Final verification and save prepared dataset
Check final shape, missing values, and duplicates, then save a separate CSV file. The original dataset is not overwritten.

In [ ]:
print("Final prepared shape:", df_prepared.shape)
print("\nRemaining missing values:", int(df_prepared.isna().sum().sum()))
print("Remaining duplicate rows:", int(df_prepared.duplicated().sum()))

display(df_prepared.head())

OUTPUT_FILE = "titanic_prepared.csv"
df_prepared.to_csv(OUTPUT_FILE, index=False)
print(f"Prepared dataset saved as: {OUTPUT_FILE}")

# In Google Colab, uncomment the next two lines to download the file:
# from google.colab import files
# files.download(OUTPUT_FILE)

## 12. Conclusion
The Titanic passenger dataset was inspected to understand its rows, columns, data types, descriptive statistics, and missing values. Missing age values were filled using group-wise medians, and missing embarkation categories were filled using the mode. The highly incomplete deck column and redundant/derived columns were removed with justification. Exact duplicate rows and category consistency were checked. EDA was performed using histograms and count plots. Nominal categorical variables were one-hot encoded, and selected numerical features were standardised using `StandardScaler`. The final prepared dataset was verified and saved as `titanic_prepared.csv`.

**No machine learning prediction model was trained**, as required by the activity.

## Submission checklist
- [ ] Run all notebook cells from top to bottom.
- [ ] Check that charts and printed outputs are visible.
- [ ] Download `titanic_prepared.csv` after running the final cell.
- [ ] Add your name, roll number, class, and submission date to the report.
- [ ] Read the output values and write EDA observations in your own words.